In [1]:
import pandas as pd
import numpy as np

# 1. Load the processed data we just created
print("Loading the processed master table...")
df = pd.read_csv('../data/processed/Master_Analytical_Table.csv')

# 2. Clean the text column
# We fill missing reviews with empty strings and make everything lowercase so our text search doesn't miss anything.
df['review_comment_message'] = df['review_comment_message'].fillna('').str.lower()

# 3. Define our keyword dictionaries (Translating Portuguese context to Business Categories)
logistics_keywords = ['atraso', 'demora', 'correios', 'entregue', 'chegou', 'prazo', 'nunca']
product_keywords = ['quebrado', 'defeito', 'ruim', 'péssimo', 'qualidade', 'estragado', 'diferente']

# 4. Build the AI categorization function
def categorize_review(text, score):
    if text == '':
        return 'No Comment'
    
    # We mainly care about categorized complaints for low scores (1, 2, or 3 stars)
    if score <= 3:
        if any(word in text for word in logistics_keywords):
            return 'Logistics & Delivery Issue'
        elif any(word in text for word in product_keywords):
            return 'Product Quality Issue'
        else:
            return 'Uncategorized Complaint'
    else:
        return 'Positive/Neutral Feedback'

# 5. Apply the function to create a new highly valuable column
print("Running text analysis on 100,000+ reviews... (This might take a few seconds)")
df['sentiment_category'] = df.apply(lambda row: categorize_review(row['review_comment_message'], row['review_score']), axis=1)

# Let's see the breakdown!
print("\n--- Complaint Breakdown ---")
print(df['sentiment_category'].value_counts())

# 6. Save the final enriched dataset for your Dashboard
output_path = '../data/processed/Final_Dashboard_Data.csv'
df.to_csv(output_path, index=False)
print(f"\nNLP feature engineering complete! Saved final data to: {output_path}")


Loading the processed master table...
Running text analysis on 100,000+ reviews... (This might take a few seconds)

--- Complaint Breakdown ---
sentiment_category
No Comment                    64283
Positive/Neutral Feedback     29190
Uncategorized Complaint        9889
Logistics & Delivery Issue     5396
Product Quality Issue          1438
Name: count, dtype: int64

NLP feature engineering complete! Saved final data to: ../data/processed/Final_Dashboard_Data.csv
